# Evaluación cuantitativa
En Colab sube Entradas_Originales.zip y Paquete_Recuperados.zip. Se comparan las muestras en la misma escala de amplitud, sin normalización independiente.

In [5]:
from pathlib import Path
import numpy as np
from scipy.io.wavfile import read
import csv,zipfile
BASE=Path.cwd()
for archive,folder,names in [('Entradas_Originales.zip','Originales',[f'secret{i}.wav' for i in range(1,6)]),('Paquete_Recuperados.zip','Recuperados',[f'recovered{i}.wav' for i in range(1,6)])]:
    if not (BASE/folder/names[0]).exists():
        (BASE/folder).mkdir(exist_ok=True)
        with zipfile.ZipFile(BASE/archive) as z:
            for name in names: (BASE/folder/name).write_bytes(z.read(name))
def leer(path):
    fs,x=read(path)
    if np.issubdtype(x.dtype,np.signedinteger): x=x.astype(float)/(-np.iinfo(x.dtype).min)
    elif x.dtype==np.uint8: x=(x.astype(float)-128)/128
    else: x=x.astype(float)
    return fs,x
rows=[]
for i in range(1,6):
    fs,x=leer(BASE/'Originales'/f'secret{i}.wav')
    fr,y=leer(BASE/'Recuperados'/f'recovered{i}.wav')
    if fs!=fr or x.shape!=y.shape: raise ValueError('Señales incompatibles')
    error=np.sum((x-y)**2)
    snr=10*np.log10(np.sum(x*x)/error) if error else np.inf
    rho=np.corrcoef(x,y)[0,1]
    rows.append((i,snr,rho))
    print(f'Secret {i}: SNR={snr:.6f} dB | Pearson={rho:.15f}')
(BASE/'Resultados').mkdir(exist_ok=True)
with (BASE/'Resultados/metricas.csv').open('w',newline='') as h:
    w=csv.writer(h); w.writerow(['mensaje','SNR_dB','Pearson']); w.writerows(rows)


Secret 1: SNR=152.419531 dB | Pearson=1.000000000000000
Secret 2: SNR=152.208507 dB | Pearson=1.000000000000000
Secret 3: SNR=152.409525 dB | Pearson=0.999999999999999
Secret 4: SNR=152.373829 dB | Pearson=1.000000000000000
Secret 5: SNR=152.270461 dB | Pearson=0.999999999999999
